# UNHD Part A -- InternVL3.5-8B (via vLLM)
Same evaluation set, same fixed OCR prompt/decoding policy as `qwen3vl_vllm.ipynb` and `minicpmv45_vllm.ipynb` (see `research_log.md` Section 12), same notebook shape (`PROJECT_HANDOFF.md` Section 5). MiniCPM-V 4.5 was abandoned (Section 15 of the log) after producing English commentary + fabricated Urdu content regardless of prompt-delivery method -- this notebook is the replacement, per the proposal's original three-model open-source lineup.

**What's different here, and why:**
- OpenGVLab's own vLLM deployment guide for this model recommends >=20GB VRAM (A100/L40S/H100-class). A T4 has ~14.5GB usable, so unlike Qwen3-VL (confirmed bitsandbytes-quantizable in vLLM) this notebook goes straight to a **pre-quantized AWQ checkpoint** as the primary path, rather than trying full precision first and quantizing only if needed. AWQ is a long-established, mainline vLLM quantization backend (much more broadly tested across custom/trust_remote_code VLM architectures than bitsandbytes was for MiniCPM), which is why it's the primary approach here rather than the fallback-of-last-resort.
- **The AWQ checkpoint used (`cyankiwi/InternVL3_5-8B-AWQ-4bit`) is a third-party community quantization, not an official OpenGVLab release** -- unlike MiniCPM's official `-int4` checkpoint. OpenGVLab has not (as of this notebook) shipped an official InternVL3.5-8B AWQ checkpoint themselves. This is a different, additional caveat on top of the usual "fallback path is unverified" one: correctness of the quantization itself is not vouched for by the original model authors.
- InternVL's prompt format uses a single `<image>` placeholder token inside a ChatML-style template (per vLLM's own official multi-image example for this model family), not MiniCPM's `(<image>./</image>)` syntax. The primary approach here uses vLLM's generic `llm.chat()` with an `image_url` content block -- the same calling convention that worked cleanly for Qwen3-VL -- with a fallback to the manual `<image>` + `apply_chat_template` approach from vLLM's own example if `llm.chat()` doesn't handle the image correctly.

**Lesson carried forward from MiniCPM (read this before troubleshooting a similar failure):** if this model produces English commentary and/or content unrelated to the actual handwriting, do **not** immediately assume it's a prompt-delivery problem and start trying alternate templates -- that exact assumption cost real GPU time on MiniCPM and turned out to be wrong (both delivery methods failed identically). Run the non-OCR vision-perception sanity check (in Cell 7's markdown) *first* to check whether the model is perceiving the image at all before touching the prompt again.

## Cell 0 -- HF token setup (READ FIRST)
Add your token as a Kaggle Secret named `HF_TOKEN` via **Add-ons -> Secrets**, toggled on for this notebook. The AWQ checkpoint used here is public/ungated, so this is a defensive fallback only -- harmless if no secret is set.

## Cell 1 -- Install vLLM and dependencies
Same reasoning as the previous two notebooks: let vLLM manage its own compatible torch build rather than fighting Kaggle's preinstalled versions.

**Carried forward pre-emptively from the MiniCPM notebook** (research_log.md Section 15): three separate missing CUDA-13 native libraries (`libnvJitLink.so.13`, `libnvrtc-builtins.so.13.0`, `libcudart.so.13`) were hit across earlier runs on this same Kaggle+vLLM combination, each needing its own `nvidia-*` pip package installed explicitly. Rather than wait to hit each one again one at a time, all three fixes are included here from the start. If a *fourth* `libXXX.so.13: cannot open shared object file` error shows up, the fix is the same pattern: search "pip package libXXX.so.13", install the matching unversioned `nvidia-*` wheel.

In [ ]:
!pip install -q -U vllm
!pip install -q -U autoawq                    # AWQ quantization backend
!pip install -q -U nvidia-cuda-nvrtc           # ships libnvrtc-builtins.so.13.0
!pip install -q -U nvidia-cuda-runtime         # ships libcudart.so.13
!pip install -q -U "transformers>=4.52.1"      # InternVL3.5-8B's stated minimum
!pip install -q pillow

import ctypes
import glob
import os
import site

# Same T4 allocator-fragmentation fix as both earlier notebooks. Must be set before torch
# initializes its CUDA allocator, so do it before anything imports torch/vllm.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Preload libnvJitLink.so.13 defensively -- AWQ kernels and bitsandbytes (if the fallback path
# is ever used) both may need CUDA-13 native libraries not always on the default loader path.
nvjitlink_candidates = sorted(set(
    path
    for sp in site.getsitepackages() + [site.getusersitepackages()]
    for path in glob.glob(os.path.join(sp, "nvidia", "*", "lib", "libnvJitLink.so.13"))
))
for candidate in nvjitlink_candidates:
    try:
        ctypes.CDLL(candidate, mode=ctypes.RTLD_GLOBAL)
        print(f"Preloaded {candidate} into the process (RTLD_GLOBAL).")
        break
    except OSError as e:
        print(f"Could not preload {candidate}: {e}")

nvidia_lib_dirs = set()
for sp in site.getsitepackages() + [site.getusersitepackages()]:
    nvidia_lib_dirs.update(glob.glob(os.path.join(sp, "nvidia", "*", "lib")))
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import vllm
print("vLLM version:", vllm.__version__)

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets and set as environment variable.")
except Exception as e:
    print("No HF_TOKEN secret found -- proceeding without auth (model is public/ungated).")
    print(f"Error was: {e}")


## Cell 2 -- Locate dataset automatically
Identical to the Qwen3-VL notebook -- same dataset, same Kaggle input mount, same auto-detect logic so it doesn't matter what the dataset gets named/versioned as.

In [ ]:
def find_dataset_root():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "evaluation_set.csv" in files:
            return root
    raise FileNotFoundError("Could not find evaluation_set.csv anywhere under /kaggle/input")

DATASET_ROOT = find_dataset_root()
CSV_PATH = os.path.join(DATASET_ROOT, "evaluation_set.csv")
IMAGES_DIR = os.path.join(DATASET_ROOT, "images")

print("CSV:", CSV_PATH)
print("Images dir:", IMAGES_DIR)
print("Images found:", len(os.listdir(IMAGES_DIR)))


## Cell 3 -- Load evaluation set, sanity-check against images on disk
Identical to the Qwen3-VL notebook.

In [ ]:
import csv

with open(CSV_PATH, encoding="utf-8-sig") as f:
    eval_rows = list(csv.DictReader(f))

print(f"Evaluation set rows: {len(eval_rows)}")

def image_filename(row):
    return row["image_path"].replace("\\", "/").rsplit("/", 1)[-1]

missing = [r for r in eval_rows if not os.path.isfile(os.path.join(IMAGES_DIR, image_filename(r)))]
print(f"Missing images: {len(missing)}")
if missing:
    print("Examples:", [image_filename(r) for r in missing[:5]])
    print("STOP -- investigate before running the full pipeline.")


## Cell 4 -- CER/WER functions (copied verbatim from the Qwen3-VL notebook)
Per `PROJECT_HANDOFF.md` Section 7: copy this exact code for any new model notebook rather than reimplementing, to guarantee comparability. Not one character changed from `qwen3vl_vllm.ipynb`.

In [ ]:
import unicodedata

def edit_distance(a, b):
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            curr[j] = min(prev[j] + 1, curr[j-1] + 1, prev[j-1] + cost)
        prev = curr
    return prev[m]

def strip_diacritics(text):
    normalized = unicodedata.normalize("NFC", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")

def clean_text(text):
    return unicodedata.normalize("NFC", (text or "").strip())

def cer(pred, gt):
    gt = clean_text(gt); pred = clean_text(pred)
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt):
    gt_words = clean_text(gt).split(); pred_words = clean_text(pred).split()
    if len(gt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return edit_distance(pred_words, gt_words) / len(gt_words)

def compute_metrics(pred, gt):
    cer_sens = cer(pred, gt)
    wer_sens = wer(pred, gt)
    pred_stripped = strip_diacritics(pred)
    gt_stripped = strip_diacritics(gt)
    cer_insens = cer(pred_stripped, gt_stripped)
    wer_insens = wer(pred_stripped, gt_stripped)
    return cer_sens, cer_insens, wer_sens, wer_insens

assert edit_distance("kitten", "sitting") == 3
print("Metric functions OK")


## Cell 5 -- Load InternVL3.5-8B with vLLM
**Primary path:** `cyankiwi/InternVL3_5-8B-AWQ-4bit`, a third-party AWQ-quantized checkpoint (~5GB), loaded with `quantization="awq"`. AWQ is one of vLLM's most mature, broadly-supported quantization backends -- much less likely to hit the kind of architecture-specific quantization friction MiniCPM did with bitsandbytes.

**Fallback path:** the official full-precision `OpenGVLab/InternVL3_5-8B` checkpoint with in-flight bitsandbytes quantization, if the AWQ checkpoint fails to load. OpenGVLab's own hardware guidance (>=20GB VRAM) suggests this fallback may not fit a T4 at all -- if both paths fail, that's a real signal this model doesn't fit this hardware, not a bug to keep chasing.

**Caveat specific to this notebook:** the AWQ checkpoint is a community quantization, not released by OpenGVLab themselves. Treat Cell 7's sanity test as a real correctness check for this reason specifically, not just a formality.

Same T4 memory-headroom settings as both earlier notebooks, same reasoning (vision-encoder activation memory isn't covered by the KV-cache budget).

In [ ]:
from vllm import LLM, SamplingParams

PRIMARY_MODEL_ID = "cyankiwi/InternVL3_5-8B-AWQ-4bit"   # third-party AWQ quantization
FALLBACK_MODEL_ID = "OpenGVLab/InternVL3_5-8B"           # official, full precision, needs in-flight bnb

COMMON_KWARGS = dict(
    dtype="float16",              # T4 (compute capability 7.5) does not support bfloat16 --
                                   # same issue hit with both earlier models.
    max_model_len=2048,           # generous for a short prompt + one handwriting line + 128
                                   # response tokens; kept conservative given MiniCPM's dummy-
                                   # profiling OverflowError was tied to max_model_len sizing.
    trust_remote_code=True,
    mm_processor_cache_gb=0,      # current vLLM's way of disabling the multimodal preprocessor
                                   # cache (see research_log.md Section 15 / MiniCPM notebook for
                                   # why disable_mm_preprocessor_cache=True no longer works).
    limit_mm_per_prompt={"image": 1},
    gpu_memory_utilization=0.75,
    max_num_batched_tokens=2048,
)

LOADED_MODEL_ID = None
USED_FALLBACK = False

try:
    llm = LLM(model=PRIMARY_MODEL_ID, quantization="awq", **COMMON_KWARGS)
    LOADED_MODEL_ID = PRIMARY_MODEL_ID
    print(f"Loaded AWQ checkpoint: {PRIMARY_MODEL_ID}")
except Exception as e:
    print(f"AWQ checkpoint failed to load:\n  {e!r}\n")
    print(f"Falling back to in-flight bitsandbytes quantization of {FALLBACK_MODEL_ID} ...")
    llm = LLM(
        model=FALLBACK_MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        **COMMON_KWARGS,
    )
    LOADED_MODEL_ID = FALLBACK_MODEL_ID
    USED_FALLBACK = True
    print(f"Loaded with in-flight bitsandbytes quantization: {FALLBACK_MODEL_ID}")

OCR_PROMPT = (
    "This image contains a single line of handwritten Urdu text in the Nastaliq script. "
    "Transcribe exactly what is written, character for character, including any diacritics. "
    "Output ONLY the transcribed Urdu text -- no translation, no explanation, no extra commentary."
)

# Same decoding policy as every other Part A model (research_log.md Section 12): deterministic,
# plus the standing repetition_penalty policy from the Qwen3-VL diagnosis.
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=128,
    repetition_penalty=1.05,
)

print("vLLM engine loaded. LOADED_MODEL_ID =", LOADED_MODEL_ID, "| USED_FALLBACK =", USED_FALLBACK)


## Cell 6 -- Helper functions: prompt template + image loading (batched)
**Primary approach:** vLLM's generic `llm.chat()` with a system-role message (the fixed OCR instruction) and a user-role message containing an `image_url` content block per conversation -- the exact calling convention that worked cleanly for Qwen3-VL, tried first here because InternVL has long-established native vLLM support and this is the most standard interface. Built to accept a *list* of image paths and return a list of predictions in the same order, matching the batched pattern the checkpointed run in Cell 9 depends on.

**Fallback:** if `llm.chat()` raises on a real batch, switch (once, for the rest of the run) to vLLM's own documented manual approach for this model family: a single `<image>` placeholder token inside a ChatML template, built with `apply_chat_template` and passed to `llm.generate()` with `multi_modal_data`, matching vLLM's official multi-image example for InternVL.

**If both produce English commentary and/or unrelated content:** per the note at the top of this notebook, treat that as evidence pointing away from prompt delivery, not toward it -- run the vision-perception check in Cell 7 before trying a third prompt variant.

In [ ]:
from PIL import Image
import base64
from io import BytesIO
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(LOADED_MODEL_ID, trust_remote_code=True)

def load_image_and_b64(img_path):
    image = Image.open(img_path).convert("RGB")
    buf = BytesIO()
    image.save(buf, format="JPEG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return image, b64

def build_conversation(b64):
    return [
        {"role": "system", "content": OCR_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ],
        },
    ]

FALLBACK_PROMPT_TEXT = tokenizer.apply_chat_template(
    [{"role": "user", "content": f"<image>\n{OCR_PROMPT}"}],
    tokenize=False, add_generation_prompt=True,
)

def build_fallback_input(image):
    return {"prompt": FALLBACK_PROMPT_TEXT, "multi_modal_data": {"image": image}}

USE_CHAT_API = True   # once flipped False (by a real failure below), stays False for the rest
                       # of the run -- avoids silently alternating between two different calling
                       # conventions mid-run, which would make results incomparable within
                       # themselves, let alone against the other Part A models.

def generate_batch(img_paths):
    """Returns predictions in the same order as img_paths. Tries llm.chat() first; permanently
    switches to the manual-template llm.generate() fallback on the first hard failure."""
    global USE_CHAT_API
    images_b64 = [load_image_and_b64(p) for p in img_paths]

    if USE_CHAT_API:
        try:
            conversations = [build_conversation(b64) for _, b64 in images_b64]
            outputs = llm.chat(conversations, sampling_params=sampling_params)
            return [o.outputs[0].text.strip() for o in outputs]
        except Exception as e:
            print(f"llm.chat() failed on a real batch ({e!r}) -- switching permanently to the "
                  f"manual <image>-placeholder fallback for the rest of this run.")
            USE_CHAT_API = False

    fallback_inputs = [build_fallback_input(image) for image, _ in images_b64]
    outputs = llm.generate(fallback_inputs, sampling_params=sampling_params)
    return [o.outputs[0].text.strip() for o in outputs]


## Cell 7 -- Single-image sanity test (READ THE OUTPUT BEFORE CONTINUING)
Same purpose as every other Part A notebook. Check specifically for:
- Output is actual Urdu text -- not empty, not English commentary, not a literal `<image>` tag echoed back.
- The Urdu content plausibly corresponds to the ground truth (some CER is fine -- this is a hard task -- but it should be recognizably related, not a different sentence).
- If `USED_FALLBACK` is `True` (Cell 5's AWQ path failed), or if `USE_CHAT_API` prints `False` after this cell (Cell 6's `llm.chat()` path failed), check this output especially carefully.

**If the output has an English preamble and/or content unrelated to the image** (the exact MiniCPM failure pattern), run the vision-perception probe below *before* trying a different prompt template:
```python
diag_conv = [{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{load_image_and_b64(test_path)[1]}"}},
    {"type": "text", "text": "Describe what you see in this image in one sentence. Do not attempt to read or transcribe any text."},
]}]
print(llm.chat([diag_conv], sampling_params=SamplingParams(temperature=0, max_tokens=64))[0].outputs[0].text.strip())
```
If that correctly describes a line of handwritten Urdu script, the vision pathway is fine and this model just can't reliably read Nastaliq -- a real finding. If it describes something generic or wrong, the image isn't reaching the model correctly, which is a different and more fixable problem.

In [ ]:
test_row = eval_rows[0]
test_path = os.path.join(IMAGES_DIR, image_filename(test_row))

test_pred = generate_batch([test_path])[0]

print("Ground truth:", test_row["text"])
print("Prediction:  ", test_pred)
print()
print("CER (sensitive):", cer(test_pred, test_row["text"]))
print("USE_CHAT_API after this call:", USE_CHAT_API)


## Cell 8 -- Timed mini-batch (12 images)
Same purpose as both earlier notebooks: get a real per-image throughput number before committing to the full 2,465-image run.

In [ ]:
import time

mini_batch = eval_rows[:12]
mini_paths = [os.path.join(IMAGES_DIR, image_filename(r)) for r in mini_batch]

start = time.time()
mini_preds = generate_batch(mini_paths)
elapsed = time.time() - start

print(f"{len(mini_batch)} images in {elapsed:.1f}s -> {elapsed/len(mini_batch):.2f}s/image")
print(f"Projected time for full {len(eval_rows)}-image run: {elapsed/len(mini_batch)*len(eval_rows)/60:.1f} minutes")

for row, pred in zip(mini_batch[:3], mini_preds[:3]):
    print("---")
    print("GT:  ", row["text"])
    print("PRED:", pred)


## Cell 9 -- Full run with checkpointing
Identical structure and resume logic to both earlier notebooks: incremental writes, resumable by `file_id`, one row per image with prediction + all four metrics + any batch-level error.

In [ ]:
import pandas as pd

OUTPUT_CSV = "internvl35_vllm_predictions.csv"
BATCH_SIZE = 16
FIELDNAMES = ["file_id", "sentence_id", "writer_id", "ground_truth", "prediction",
              "cer_sensitive", "cer_insensitive", "wer_sensitive", "wer_insensitive", "error"]

done_ids = set()
if os.path.isfile(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    done_ids = set(existing["file_id"].astype(str))
    print(f"Resuming: {len(done_ids)} rows already completed.")
else:
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()
    print("Starting fresh run.")

remaining_rows = [r for r in eval_rows if str(r.get("file_id", image_filename(r))) not in done_ids]
print(f"Remaining: {len(remaining_rows)} / {len(eval_rows)}")

for batch_start in range(0, len(remaining_rows), BATCH_SIZE):
    batch_rows = remaining_rows[batch_start:batch_start + BATCH_SIZE]
    batch_paths = [os.path.join(IMAGES_DIR, image_filename(r)) for r in batch_rows]

    try:
        predictions = generate_batch(batch_paths)
        batch_errors = [""] * len(batch_rows)
    except Exception as e:
        # Whole-batch failure: record the error against every row in this batch rather than
        # silently dropping them, and score each as maximally wrong (cer=wer=1.0) so the batch
        # still counts in the denominator instead of vanishing from the aggregate.
        predictions = [""] * len(batch_rows)
        batch_errors = [str(e)] * len(batch_rows)

    rows_to_write = []
    for row, pred, err in zip(batch_rows, predictions, batch_errors):
        gt = row["text"]
        if err:
            cs = ci = ws = wi = 1.0
        else:
            cs, ci, ws, wi = compute_metrics(pred, gt)
        rows_to_write.append({
            "file_id": row.get("file_id", image_filename(row)),
            "sentence_id": row["sentence_id"],
            "writer_id": row["writer_id"],
            "ground_truth": gt,
            "prediction": pred,
            "cer_sensitive": cs, "cer_insensitive": ci,
            "wer_sensitive": ws, "wer_insensitive": wi,
            "error": err,
        })

    with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writerows(rows_to_write)

    done_so_far = batch_start + len(batch_rows)
    print(f"{done_so_far}/{len(remaining_rows)} done"
          + (f"  -- BATCH ERROR: {batch_errors[0][:120]}" if batch_errors[0] else ""))

print("Full run complete.")


## Cell 10 -- Aggregate results
Identical structure to both earlier notebooks -- straight mean of each per-row metric column, plus the same integrity checks (duplicate `file_id`, empty predictions, logged batch errors).

In [ ]:
df = pd.read_csv(OUTPUT_CSV, encoding="utf-8")
df["prediction"] = df["prediction"].fillna("")
df["error"] = df["error"].fillna("")

print("=== Integrity checks ===")
print("Rows:", len(df))
print("Duplicate file_id:", df["file_id"].duplicated().sum())
print("Rows with non-empty error:", (df["error"] != "").sum())
print("Empty predictions:", (df["prediction"].str.strip() == "").sum())
gt_check = df.groupby("sentence_id")["ground_truth"].nunique()
print("sentence_ids with >1 distinct ground truth:", (gt_check > 1).sum())
print()

print("=== Aggregate metrics ===")
print(f"Images evaluated: {len(df)}")
print(f"CER (diacritic-sensitive):   {df['cer_sensitive'].mean():.4f}")
print(f"CER (diacritic-insensitive): {df['cer_insensitive'].mean():.4f}")
print(f"WER (diacritic-sensitive):   {df['wer_sensitive'].mean():.4f}")
print(f"WER (diacritic-insensitive): {df['wer_insensitive'].mean():.4f}")
print(f"Empty predictions: {(df['prediction'].str.strip() == '').sum()} / {len(df)}")


## Cell 11 -- Error analysis
Same breakdown as both earlier notebooks: failure-rate buckets, length-vs-CER correlation, worst-CER examples, repetition-loop check.

In [ ]:
print("CER >= 1.0 (worse than blank):", (df['cer_sensitive'] >= 1.0).sum(),
      f"({(df['cer_sensitive'] >= 1.0).mean()*100:.2f}%)")
print("CER >= 0.9 (near-total failure):", (df['cer_sensitive'] >= 0.9).sum(),
      f"({(df['cer_sensitive'] >= 0.9).mean()*100:.2f}%)")

df["gt_len"] = df["ground_truth"].str.len()
df["pred_len"] = df["prediction"].str.len()
long_preds = df[df["pred_len"] > df["gt_len"] * 2]
print(f"Predictions >2x ground-truth length (possible repetition loop): {len(long_preds)}",
      f"({len(long_preds)/len(df)*100:.2f}%)")
print("Length-vs-CER correlation (r):", df["gt_len"].corr(df["cer_sensitive"]))
print()

print("=== Worst 5 predictions by CER ===")
for _, row in df.sort_values("cer_sensitive", ascending=False).head(5).iterrows():
    print(f"file_id={row['file_id']}  CER={row['cer_sensitive']:.2f}")
    print("  GT:  ", row["ground_truth"])
    print("  PRED:", row["prediction"])
    print()

diacritic_drop = df["cer_sensitive"] - df["cer_insensitive"]
big_drop = (diacritic_drop > 0.5)
print(f"Sentences with >50% CER drop when diacritics are stripped: {big_drop.sum()} / {len(df)}")


## Cell 12 -- Save summary

In [ ]:
summary_lines = [
    "INTERNVL3.5-8B (vLLM) BASELINE -- RESULTS SUMMARY",
    "=" * 50,
    f"Model checkpoint used: {LOADED_MODEL_ID}{' (in-flight bitsandbytes fallback)' if USED_FALLBACK else ''}",
    f"Prompt delivery: {'llm.chat() + image_url' if USE_CHAT_API else 'manual <image> placeholder + llm.generate()'}",
    f"Images evaluated: {len(df)}",
    f"CER (diacritic-sensitive):   {df['cer_sensitive'].mean():.4f}",
    f"CER (diacritic-insensitive): {df['cer_insensitive'].mean():.4f}",
    f"WER (diacritic-sensitive):   {df['wer_sensitive'].mean():.4f}",
    f"WER (diacritic-insensitive): {df['wer_insensitive'].mean():.4f}",
    f"Empty predictions: {(df['prediction'].str.strip() == '').sum()} / {len(df)}",
    f"Length-vs-CER correlation (r): {df['gt_len'].corr(df['cer_sensitive']):.4f}",
]

with open("internvl35_vllm_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines) + "\n")

print("\n".join(summary_lines))
